In [ ]:
from ast import literal_eval

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import average_precision_score, roc_auc_score


In [ ]:
def signal_features(model, s):
    match model:
        case "Change point":
            return list(range(s))
        case "Interaction":
            return list(range(2*s))
        case "Linear":
            return list(range(s, 2*s))
        case "Mixed":
            return list(range(2*s))
        case _:
            raise ValueError(f"Unknown data model {model}.")

In [ ]:
results = pd.concat((
    pd.read_csv("results-depth=None+2026-07-17+170137.csv"),
    pd.read_csv("results-depth=None+2026-07-22+181230.csv"),
), ignore_index=True).replace({
    "LSS confounding": "Change point",
    "Forest DML (econML)": "Forest DML",
})
results['MSE / MSE_0'] = results['MSE'] / results['MSE_0']
results['PEHE / PEHE_0'] = np.sqrt(results['MSE / MSE_0'])
results['Feature importance'] = results['Feature importance'].map(literal_eval).map(np.array)

In [ ]:
for i in results.index:
    s = results.loc[i, "s"]
    model = results.loc[i, "model"]
    signal_feat = signal_features(model, s)
    signal_indicator = np.zeros(5*s)
    signal_indicator[signal_feat] = 1
    feat_importance = results.loc[i, 'Feature importance']
    results.loc[i, 'Heterogeneity feature importance'] = feat_importance[signal_feat].sum()
    results.loc[i, 'ROC-AUC'] = roc_auc_score(signal_indicator, feat_importance)
    results.loc[i, 'avg precision'] = average_precision_score(signal_indicator, feat_importance)

In [ ]:
results

## Output formatting

In [ ]:
output_format = r"{mean:.3f} ± {se:.3f}"
model_order = ['Interaction', 'Change point', 'Linear', 'Mixed']
method_order = ['IntCF', 'Forest DML', 'Causal Forest', 'GRF']
plot_order = [[0, 0], [0, 1], [1, 0], [1, 1]]
mpl.rcParams['font.family'] = 'Frutiger Next LT W1G'
sns.set_context("paper")
figsize = (8, 6)
# sns.set_context("poster")
# figsize = (16, 12)

## Combined plots

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=figsize, layout='constrained')

for model, pos in zip(model_order, plot_order):
    ax = axs[pos[0], pos[1]]
    filtered = results[results['model'] == model]
    sns.lineplot(
        filtered, x='s', y='PEHE / PEHE_0', errorbar='se',
        hue='Estimation method', hue_order=method_order,
        ax=ax, legend=(pos==[0, 1])
    )
    if pos[0] == 0:
        ax.set_xlabel(None)
    if pos[1] == 0:
        ax.set_ylabel(r"$\frac{\operatorname{PEHE}}{\operatorname{PEHE}_0}$")
    else:
        ax.set_ylabel(None)
        sec_ax = ax.secondary_yaxis('right')
        sec_ax.set_yticks(ax.get_ylim(), ["best", "worst"])
    ax.set_title(model + " model")

plt.savefig("img/paper-pehe.png", bbox_inches='tight')

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=figsize, layout='constrained')

for model, pos in zip(model_order, plot_order):
    ax = axs[pos[0], pos[1]]
    filtered = results[results['model'] == model]
    sns.lineplot(
        filtered, x='s', y='Heterogeneity feature importance', errorbar='se',
        hue='Estimation method', hue_order=["IntCF", "IntCF (without validation)"],
        ax=ax, legend=(pos==[0, 1])
    )
    if pos[0] == 0:
        ax.set_xlabel(None)
    if pos[1] == 0:
        ax.set_ylabel("Feature importance")
    else:
        ax.set_ylabel(None)
        sec_ax = ax.secondary_yaxis('right')
        sec_ax.set_yticks(ax.get_ylim(), ["worst", "best"])
    ax.set_title(model + " model")

plt.savefig("img/intcf-feat_importance.png", bbox_inches='tight')

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=figsize, layout='constrained')

for model, pos in zip(model_order, plot_order):
    ax = axs[pos[0], pos[1]]
    filtered = results[results['model'] == model]
    sns.lineplot(
        filtered, x='s', y='ROC-AUC', errorbar='se',
        hue='Estimation method', hue_order=method_order,
        ax=ax, legend=(pos==[0, 1])
    )
    if pos[0] == 0:
        ax.set_xlabel(None)
    if pos[1] == 0:
        ax.set_ylabel("ROC-AUC")
    else:
        ax.set_ylabel(None)
        sec_ax = ax.secondary_yaxis('right')
        sec_ax.set_yticks(ax.get_ylim(), ["worst", "best"])
    ax.set_title(model + " model")

plt.savefig("img/paper-roc_auc.png", bbox_inches='tight')

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=figsize, layout='constrained')

for model, pos in zip(model_order, plot_order):
    ax = axs[pos[0], pos[1]]
    filtered = results[results['model'] == model]
    sns.lineplot(
        filtered, x='s', y='avg precision', errorbar='se',
        hue='Estimation method', hue_order=method_order,
        ax=ax, legend=(pos==[0, 1])
    )
    if pos[0] == 0:
        ax.set_xlabel(None)
    if pos[1] == 0:
        ax.set_ylabel("Precision-Recall AUC")
    else:
        ax.set_ylabel(None)
        sec_ax = ax.secondary_yaxis('right')
        sec_ax.set_yticks(ax.get_ylim(), ["worst", "best"])
    ax.set_title(model + " model")

# plt.savefig("img/all-precision_recall.png", bbox_inches='tight')

## Graphs for appendix

In [ ]:
method_order = ['IntCF', 'Forest DML', 'Causal Forest', 'GRF', 'IntCF (without validation)']

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=figsize, layout='constrained')

for model, pos in zip(model_order, plot_order):
    ax = axs[pos[0], pos[1]]
    filtered = results[results['model'] == model]
    sns.lineplot(
        filtered, x='s', y='PEHE / PEHE_0', errorbar='se',
        hue='Estimation method', hue_order=method_order,
        ax=ax, legend=(pos==[0, 1])
    )
    if pos[0] == 0:
        ax.set_xlabel(None)
    if pos[1] == 0:
        ax.set_ylabel(r"$\frac{\operatorname{PEHE}}{\operatorname{PEHE}_0}$")
    else:
        ax.set_ylabel(None)
        sec_ax = ax.secondary_yaxis('right')
        sec_ax.set_yticks(ax.get_ylim(), ["best", "worst"])
    ax.set_title(model + " model")

plt.savefig("img/all-pehe.png", bbox_inches='tight')

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=figsize, layout='constrained')

for model, pos in zip(model_order, plot_order):
    ax = axs[pos[0], pos[1]]
    filtered = results[results['model'] == model]
    sns.lineplot(
        filtered, x='s', y='Heterogeneity feature importance', errorbar='se',
        hue='Estimation method', hue_order=method_order,
        ax=ax, legend=(pos==[0, 0])
    )
    if pos[0] == 0:
        ax.set_xlabel(None)
    if pos[1] == 0:
        ax.set_ylabel("Feature importance")
    else:
        ax.set_ylabel(None)
        sec_ax = ax.secondary_yaxis('right')
        sec_ax.set_yticks(ax.get_ylim(), ["worst", "best"])
    ax.set_title(model + " model")

plt.savefig("img/all-feat_importance.png", bbox_inches='tight')

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=figsize, layout='constrained')

for model, pos in zip(model_order, plot_order):
    ax = axs[pos[0], pos[1]]
    filtered = results[results['model'] == model]
    sns.lineplot(
        filtered, x='s', y='ROC-AUC', errorbar='se',
        hue='Estimation method', hue_order=method_order,
        ax=ax, legend=(pos==[0, 1])
    )
    if pos[0] == 0:
        ax.set_xlabel(None)
    if pos[1] == 0:
        ax.set_ylabel("ROC-AUC")
    else:
        ax.set_ylabel(None)
        sec_ax = ax.secondary_yaxis('right')
        sec_ax.set_yticks(ax.get_ylim(), ["worst", "best"])
    ax.set_title(model + " model")

plt.savefig("img/all-roc_auc.png", bbox_inches='tight')

## Tables with all results

In [ ]:
outputs_mean = results.groupby(by=['model', 's', 'Estimation method']).mean(numeric_only=True)
outputs_se = results.groupby(by=['model', 's', 'Estimation method']).sem(numeric_only=True)

In [ ]:
metric = 'ROC-AUC'  # 'PEHE / PEHE_0', 'Heterogeneity feature importance', 'ROC-AUC', 'avg precision'
table = outputs_mean[metric].combine(outputs_se[metric], lambda mean, se: output_format.format(mean=mean, se=se)).reset_index().pivot(index=['model', 's'], columns='Estimation method', values=metric).loc[model_order, method_order]
print(table.reset_index().to_latex(index=False))

In [ ]:
table